# U-Net Apex Heatmap — Training & Evaluation

End-to-end notebook covering:
1. Synthetic chromatogram visualization
2. Chunking visualization
3. Training data generation & inspection
4. Model training with loss curve
5. Quantitative evaluation (confusion matrix, over/under-prediction)
6. Full pipeline demo on synthetic data (uses production `run_deconvolution_pipeline`)

**Primary interface:** `pipeline_viz.py`  
**Run from:** `shoulder-detection/` directory

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
import matplotlib.pyplot as plt

from pipeline_viz import (
    plot_chromatograms,
    plot_chunked_chromatograms,
    plot_training_grid,
    train_model,
    load_model,
    evaluate_model,
)
from training_data import generate_training_data, plot_training_samples
from unet_model import GCHeatmapUNet, WeightedMSELoss, MultiChannelLoss
from synthetic_chromatogram import SyntheticChromatogramGenerator, plot_synthetic
from logic.deconvolution import run_deconvolution_pipeline

# ── Control flags ─────────────────────────────────────────────────────────────
# True  → generate training data, train, and save weights to NEW_WEIGHTS_PATH
# False → load existing weights from NEW_WEIGHTS_PATH (or WEIGHTS_PATH as fallback)
TRAIN = False

# Number of output channels: 1 = apex only, 3 = apex + sigma + tau
N_CHANNELS = 1

# ── Paths ─────────────────────────────────────────────────────────────────────
WEIGHTS_PATH     = 'gc_heatmap_unet_v2.pth'   # previous baseline
NEW_WEIGHTS_PATH = 'gc_heatmap_unet_v3.pth'   # target for new training run
PROD_WEIGHTS_PATH = '../cache/gc_heatmap_unet.pth'

# ── Device ────────────────────────────────────────────────────────────────────
if torch.backends.mps.is_available():
    DEVICE = 'mps'
elif torch.cuda.is_available():
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'
print(f'Device: {DEVICE}  |  TRAIN={TRAIN}  |  N_CHANNELS={N_CHANNELS}')

## 1. Synthetic Chromatogram Visualization

Inspect the generator output with the updated noise model (colored 1/f noise + sinusoidal ripple) and the new shoulder-constrained peak placement.

In [ ]:
# Single chromatogram — detailed view with individual EMG components
gen = SyntheticChromatogramGenerator(seed=42)
result = gen.generate(num_peaks=12, merge_probability=0.5, snr=150)

n_merged = sum(1 for p in result['true_peaks'] if p['is_merged'])
print(f"Total peaks: {len(result['true_peaks'])}, merged: {n_merged}")

fig = plot_synthetic(result, engine='plotly')
fig.show()

In [ ]:
# Grid of several chromatograms — check merge rate and noise realism
fig = plot_chromatograms(n=4, seed=10, merge_probability=0.5, snr=150, engine='plotly')
fig.show()

In [ ]:
# Empirical merge rate across many chromatograms
gen = SyntheticChromatogramGenerator(seed=0)
total_peaks, total_merged = 0, 0
for _ in range(200):
    r = gen.generate(num_peaks=(8, 20), merge_probability=0.5)
    total_peaks  += len(r['true_peaks'])
    total_merged += sum(1 for p in r['true_peaks'] if p['is_merged'])
print(f'Merge rate: {total_merged/total_peaks:.1%}  ({total_merged}/{total_peaks} peaks merged)')

## 2. Chunking Visualization

Verify the chunker correctly groups merged peaks into single windows and that chunk boundaries don't split individual peaks.

In [ ]:
fig = plot_chunked_chromatograms(
    n=4, seed=10,
    merge_probability=0.5, snr=150,
    engine='plotly',
)
fig.show()

## 3. Training Data

Skipped when `TRAIN=False`.  
Val data (500 chromatograms) is always generated for evaluation in section 7.

New in v3:
- Realistic 1/f + sinusoidal noise model (`noise_mode='realistic'`)
- Random smoothing augmentation (SavGol / median / Whittaker at 70% probability)
- Shoulder-constrained peak placement (inflection-point validation)

In [ ]:
NUM_TRAIN = 5000
NUM_VAL   = 500

CHROM_KW = dict(
    num_peaks=(5, 20),
    merge_probability=0.5,
    snr=None,           # random 30-500
    baseline_type='random',
)

if TRAIN:
    print(f'Generating {N_CHANNELS}-channel training data...')
    X_train, Y_train, meta_train = generate_training_data(
        num_chromatograms=NUM_TRAIN,
        seed=42,
        smoothing_augmentation=True,
        chromatogram_kwargs=CHROM_KW,
        n_channels=N_CHANNELS,
    )
    print(f'Training: {len(X_train)} windows  X={X_train.shape}  Y={Y_train.shape}')
else:
    print('TRAIN=False — skipping training data generation')
    X_train = Y_train = meta_train = None

print(f'Generating {N_CHANNELS}-channel validation data...')
X_val, Y_val, meta_val = generate_training_data(
    num_chromatograms=NUM_VAL,
    seed=999,
    smoothing_augmentation=False,
    chromatogram_kwargs=CHROM_KW,
    n_channels=N_CHANNELS,
)
print(f'Validation: {len(X_val)} windows  X={X_val.shape}  Y={Y_val.shape}')

In [ ]:
# Inspect training windows — only meaningful when TRAIN=True
if TRAIN and X_train is not None:
    from collections import Counter
    print(f'Apex distribution: {dict(sorted(Counter(m["n_apexes"] for m in meta_train).items()))}')
    fig = plot_training_grid(X_train, Y_train, meta_train, rows=5, cols=5, seed=7, engine='plotly')
    fig.show()
    fig2 = plot_training_samples(X_train, Y_train, meta_train, n=8, engine='plotly')
    fig2.show()
else:
    # Show val set instead so there's something to look at
    from collections import Counter
    print(f'Val apex distribution: {dict(sorted(Counter(m["n_apexes"] for m in meta_val).items()))}')
    fig = plot_training_grid(X_val, Y_val, meta_val, rows=3, cols=5, seed=7, engine='plotly')
    fig.show()

## 4. Train or Load Model

`TRAIN=True` → trains from scratch and saves to `NEW_WEIGHTS_PATH`.  
`TRAIN=False` → loads `NEW_WEIGHTS_PATH` (falls back to `WEIGHTS_PATH` if not found).

In [ ]:
if TRAIN:
    EPOCHS     = 30
    BATCH_SIZE = 64
    LR         = 1e-3

    if N_CHANNELS == 3:
        loss_fn = MultiChannelLoss(peak_weight=50.0, param_weight=1.0)
        model   = GCHeatmapUNet(out_channels=3)
        print('Using MultiChannelLoss (3-channel model)')
    else:
        loss_fn = WeightedMSELoss(peak_weight=50.0)
        model   = GCHeatmapUNet()
        print('Using WeightedMSELoss (1-channel model)')

    model, history = train_model(
        X_train, Y_train,
        model=model,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
        loss_fn=loss_fn,
        device=DEVICE,
        save_path=NEW_WEIGHTS_PATH,
        verbose=True,
    )

    import plotly.graph_objects as go
    fig = go.Figure(go.Scatter(
        y=history, mode='lines+markers',
        line=dict(color='#1f77b4', width=2), marker=dict(size=5),
    ))
    fig.update_layout(
        title='Training Loss (WeightedMSE)',
        xaxis_title='Epoch', yaxis_title='Loss',
        template='plotly_white', height=350,
    )
    fig.show()

else:
    # Load existing weights — prefer NEW_WEIGHTS_PATH, fall back to WEIGHTS_PATH
    import os
    weights_to_load = NEW_WEIGHTS_PATH if os.path.isfile(NEW_WEIGHTS_PATH) else WEIGHTS_PATH
    model = load_model(weights_to_load)
    print(f'Loaded model from {weights_to_load}')

In [ ]:
# Continue training (optional — only useful when TRAIN=True)
# model, history2 = train_model(
#     X_train, Y_train,
#     model=model,
#     epochs=20,
#     batch_size=64,
#     lr=3e-4,          # lower LR for fine-tuning
#     loss_fn=WeightedMSELoss(peak_weight=50.0),
#     device=DEVICE,
#     save_path=NEW_WEIGHTS_PATH,
#     verbose=True,
# )

## 5. Evaluate Model

Classifies each 256-pt window by peak count, produces a confusion matrix, and shows representative correct / over-predicted / under-predicted examples.

In [ ]:
# evaluate_model expects Y with shape (N, 256, 1) — extract channel 0 if multi-channel
if Y_val.shape[-1] > 1:
    Y_val_ch0 = Y_val[:, :, 0:1]
else:
    Y_val_ch0 = Y_val

eval_results = evaluate_model(
    model,
    X_test=X_val,
    Y_test=Y_val_ch0,
    metadata_test=meta_val,
    heatmap_height=0.15,
    heatmap_distance=10,
    device=DEVICE,
    engine='plotly',
    max_count_label=4,
)

In [ ]:
# Compare against v2 baseline
if os.path.isfile(WEIGHTS_PATH):
    print(f'--- {WEIGHTS_PATH} (baseline) ---')
    model_v2 = load_model(WEIGHTS_PATH)
    eval_v2 = evaluate_model(
        model_v2,
        X_test=X_val, Y_test=Y_val, metadata_test=meta_val,
        heatmap_height=0.15, heatmap_distance=10,
        device=DEVICE, engine='plotly', max_count_label=4,
    )
    print(f"\nBaseline accuracy: {eval_v2['accuracy']:.4f}")
    print(f"Current accuracy:  {eval_results['accuracy']:.4f}")
else:
    print(f'{WEIGHTS_PATH} not found — skipping baseline comparison')

## 5b. End-to-End Pipeline Evaluation

Runs the **production** `run_deconvolution_pipeline()` on N synthetic chromatograms and computes per-peak quality metrics using Hungarian matching.

Metrics:
- **Precision/Recall/F1** — are we finding the right number of peaks?
- **RT error** — are fitted peaks at the right positions?
- **Sigma/Tau ratios** — are peak shapes correct?
- **Area ratios** — are peak areas accurate?
- **Failure modes** — fat, narrow, RT drift, misses, phantoms

In [ ]:
from evaluation import evaluate_pipeline, print_report, plot_error_distributions, plot_failure_breakdown, plot_worst_cases

# Use the model weights that were just trained or loaded
demo_weights = NEW_WEIGHTS_PATH if os.path.isfile(NEW_WEIGHTS_PATH) else WEIGHTS_PATH

# Run with optimized defaults (heatmap_threshold=0.44, min_prominence=0.07,
# pre_fit_signal_threshold=0.057, plus post-fit dedup and area filter)
report = evaluate_pipeline(
    num_chromatograms=200,
    seed=7777,
    weights_path=demo_weights,
    pipeline_kwargs={},  # uses optimized defaults
    chromatogram_kwargs=CHROM_KW,
)
print_report(report)

In [ ]:
plot_error_distributions(report)
plot_failure_breakdown(report)
plot_worst_cases(report, n=6)

## 5c. Before / After Comparison

Compare the old baseline defaults (high phantom rate) vs the Optuna-optimized parameters.

| Parameter | Old | Optimized |
|---|---|---|
| `heatmap_threshold` | 0.15 | **0.44** |
| `min_prominence` | 0.01 | **0.07** |
| `pre_fit_signal_threshold` | 0 | **0.057** |
| Post-fit dedup (σ factor) | — | **0.83** |
| Post-fit area filter (frac) | — | **0.12** |

In [ ]:
import json, time as _time

demo_weights = NEW_WEIGHTS_PATH if os.path.isfile(NEW_WEIGHTS_PATH) else WEIGHTS_PATH

# ── Old baseline (high phantom rate) ──────────────────────────────────────────
print('Running BASELINE (old defaults)...')
t0 = _time.time()
report_old = evaluate_pipeline(
    num_chromatograms=200, seed=7777,
    weights_path=demo_weights,
    pipeline_kwargs=dict(
        heatmap_threshold=0.15,
        min_prominence=0.01,
        pre_fit_signal_threshold=0,
    ),
    chromatogram_kwargs=CHROM_KW,
)
baseline_time = _time.time() - t0
print(f'  Done in {baseline_time:.0f}s')
print_report(report_old)

# ── Optimized (new defaults — already baked into production code) ─────────────
print('\nRunning OPTIMIZED (new defaults)...')
t0 = _time.time()
report_new = evaluate_pipeline(
    num_chromatograms=200, seed=7777,
    weights_path=demo_weights,
    pipeline_kwargs={},  # uses new defaults
    chromatogram_kwargs=CHROM_KW,
)
optimized_time = _time.time() - t0
print(f'  Done in {optimized_time:.0f}s')
print_report(report_new)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Comparison bar chart ──────────────────────────────────────────────────────
metrics = ['F1', 'Precision', 'Recall']
old_vals = [report_old.f1, report_old.precision, report_old.recall]
new_vals = [report_new.f1, report_new.precision, report_new.recall]

fig = make_subplots(rows=1, cols=2, subplot_titles=['Quality Metrics', 'Error Counts'],
                    column_widths=[0.5, 0.5])

fig.add_trace(go.Bar(name='Baseline', x=metrics, y=old_vals,
                      marker_color='#d62728', opacity=0.8), row=1, col=1)
fig.add_trace(go.Bar(name='Optimized', x=metrics, y=new_vals,
                      marker_color='#2ca02c', opacity=0.8), row=1, col=1)

counts = ['Matched', 'Phantoms', 'Misses']
old_counts = [report_old.total_matched, report_old.total_phantoms, report_old.total_misses]
new_counts = [report_new.total_matched, report_new.total_phantoms, report_new.total_misses]

fig.add_trace(go.Bar(name='Baseline', x=counts, y=old_counts,
                      marker_color='#d62728', opacity=0.8, showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(name='Optimized', x=counts, y=new_counts,
                      marker_color='#2ca02c', opacity=0.8, showlegend=False), row=1, col=2)

fig.update_layout(
    title='Baseline vs Optimized Pipeline',
    barmode='group', template='plotly_white', height=400,
)
fig.show()

# ── Summary table ─────────────────────────────────────────────────────────────
print(f'\n{"Metric":<25} {"Baseline":>12} {"Optimized":>12} {"Delta":>12}')
print('-' * 63)
rows = [
    ('F1',        report_old.f1,             report_new.f1),
    ('Precision',  report_old.precision,      report_new.precision),
    ('Recall',     report_old.recall,         report_new.recall),
    ('Matched',    report_old.total_matched,  report_new.total_matched),
    ('Phantoms',   report_old.total_phantoms, report_new.total_phantoms),
    ('Misses',     report_old.total_misses,   report_new.total_misses),
    ('Fat (σ>2)',  report_old.n_fat,          report_new.n_fat),
    ('Time (s)',   baseline_time,             optimized_time),
]
for label, v_old, v_new in rows:
    if isinstance(v_old, int):
        print(f'{label:<25} {v_old:>12d} {v_new:>12d} {v_new-v_old:>+12d}')
    else:
        print(f'{label:<25} {v_old:>12.4f} {v_new:>12.4f} {v_new-v_old:>+12.4f}')

In [ ]:
# Error distributions: old vs new
fig_old = plot_error_distributions(report_old)
fig_old.suptitle('Error Distributions — BASELINE', fontsize=14)
plt.show()

fig_new = plot_error_distributions(report_new)
fig_new.suptitle('Error Distributions — OPTIMIZED', fontsize=14)
plt.show()

## 6. Full Pipeline Demo

Runs the **production** `run_deconvolution_pipeline()` on a fresh synthetic chromatogram.

Key parameters:
- `min_prominence` — fraction of signal range below which components are discarded as noise.  
  Prevents the EMG fitter from spending time on spurious peaks in low-S/N chunks.  
  Match this to `params['peaks']['min_prominence']` in the desktop app.
- `smoothing_params` — production smoothing dict (same structure as `params['smoothing']`).  
  When `enabled=True`, the smoothed signal is fed to the U-Net; chunking and EMG fitting  
  still use the original corrected signal.

In [ ]:
import os

# Generate a synthetic chromatogram
gen = SyntheticChromatogramGenerator(seed=7)
demo_result = gen.generate(
    num_peaks=10,
    merge_probability=0.5,
    snr=150,
    baseline_type='linear',
    time_range=(0, 30),
)
time_axis = demo_result['x']
signal    = demo_result['corrected_y']

# Smoothing params for U-Net inference (production framework)
unet_smoothing = {
    'enabled': True,
    'method': 'whittaker',
    'median_enabled': True,
    'median_kernel': 5,
    'lambda': 1e1,
    'diff_order': 1,
}

# Use the model weights that were just trained or loaded
demo_weights = NEW_WEIGHTS_PATH if os.path.isfile(NEW_WEIGHTS_PATH) else WEIGHTS_PATH

deconv_result = run_deconvolution_pipeline(
    time_axis, signal,
    weights_path=demo_weights,
    smoothing_params=unet_smoothing,
)

print(f'Chunks: {len(deconv_result.chunks)}')
print(f'Components found: {len(deconv_result.components)}')
print(f'True peaks: {len(demo_result["true_peaks"])}')

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scattergl(
    x=time_axis, y=signal, mode='lines',
    line=dict(color='#333333', width=0.8), name='Signal',
))

for p in demo_result['true_peaks']:
    color = 'green' if not p['is_merged'] else 'orange'
    fig.add_trace(go.Scattergl(
        x=[p['rt']], y=[signal[p['apex_index']]], mode='markers',
        marker=dict(symbol='triangle-up', size=10, color=color),
        showlegend=False,
        text=f'True RT={p["rt"]:.2f}', hoverinfo='text',
    ))

colors = ['#e6194b','#3cb44b','#4363d8','#f58231','#911eb4',
          '#42d4f4','#f032e6','#bfef45','#fabebe','#469990']
for i, comp in enumerate(deconv_result.components):
    c = colors[i % len(colors)]
    fig.add_trace(go.Scattergl(
        x=comp.t_curve, y=comp.y_curve, mode='lines',
        line=dict(color=c, width=1.2, dash='dot'),
        name=f'EMG RT={comp.retention_time:.2f}',
    ))
    apex_idx = int(np.argmax(comp.y_curve))
    fig.add_trace(go.Scattergl(
        x=[comp.t_curve[apex_idx]], y=[comp.y_curve[apex_idx]], mode='markers',
        marker=dict(symbol='diamond', size=8, color=c,
                    line=dict(color='black', width=0.5)),
        showlegend=False,
        text=f'RT={comp.retention_time:.2f}<br>h={comp.peak_height:.1f}',
        hoverinfo='text',
    ))

fig.update_layout(
    title=f'Pipeline Demo — {len(deconv_result.components)} components '
          f'({len(demo_result["true_peaks"])} true peaks)',
    xaxis_title='Retention Time (min)', yaxis_title='Signal',
    template='plotly_white', height=450, hovermode='closest',
    legend=dict(font=dict(size=9)),
)
fig.show()

In [ ]:
true_rts   = sorted(p['rt'] for p in demo_result['true_peaks'])
fitted_rts = sorted(c.retention_time for c in deconv_result.components)
print(f'True peaks  ({len(true_rts)}): {[f"{rt:.2f}" for rt in true_rts]}')
print(f'Fitted EMG  ({len(fitted_rts)}): {[f"{rt:.2f}" for rt in fitted_rts]}')

In [ ]:
# Side-by-side: smoothed vs raw inference
deconv_raw = run_deconvolution_pipeline(
    time_axis, signal,
    weights_path=demo_weights,
    smoothing_params=None,
)
print(f'With smoothing:    {len(deconv_result.components)} components')
print(f'Without smoothing: {len(deconv_raw.components)} components')

## 7. Deploy to Production

Copy the trained weights to `cache/` for use by `logic/deconvolution.py`.

In [ ]:
import shutil

# Uncomment when satisfied with evaluation results:
# shutil.copy(NEW_WEIGHTS_PATH, PROD_WEIGHTS_PATH)
# print(f'Deployed {NEW_WEIGHTS_PATH} → {PROD_WEIGHTS_PATH}')